In [ ]:
# Drive is optional: when it is already mounted the trainer checkpoints there,
# otherwise /content keeps the run alive. No auth popup, so this cell can run
# unattended.
import os
try:
    if os.path.isdir("/content/drive/MyDrive"):
        print("drive already mounted")
except Exception as e:
    print("drive unavailable", e)


In [ ]:
# ===== Black Whole — free-GPU trainer (Colab) =====
# Dataset: public copy of the repo corpus. Checkpoints + logs: Google Drive, so
# a disconnect resumes instead of restarting. Nothing here costs money.
import json, math, os, random, time, tarfile, urllib.request
DRIVE = "/content/drive/MyDrive/blackwhole-model" if os.path.isdir("/content/drive/MyDrive") else "/content/bw"
os.makedirs(DRIVE, exist_ok=True)
CORPUS = os.path.join(DRIVE, "corpus.txt")
URL = "https://base44.app/api/apps/6ae1e734e5de94685a1f86a1/files/mp/public/6ae1e734e5de94685a1f86a1/a495c5b40_corpus.txt"
if not os.path.exists(CORPUS):
    urllib.request.urlretrieve(URL, CORPUS)
text = open(CORPUS, encoding="utf-8", errors="ignore").read()
print("corpus chars", len(text))
import torch, torch.nn as nn, torch.nn.functional as F
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
dev = "cuda" if torch.cuda.is_available() else "cpu"
chars = sorted(set(text)); stoi = {c:i for i,c in enumerate(chars)}; itos = {i:c for c,i in stoi.items()}
V = len(chars); data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(len(data)*0.97); train_d, val_d = data[:n], data[n:]
BLOCK = 256; BATCH = 32; LR = 6e-4; STEPS = int(os.environ.get("BW_STEPS", "6000"))
class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.ln1=nn.LayerNorm(d); s.at=nn.MultiheadAttention(d,h,batch_first=True,dropout=0.1)
        s.ln2=nn.LayerNorm(d); s.ml=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d),nn.Dropout(0.1))
    def forward(s,x):
        a,_=s.at(s.ln1(x),s.ln1(x),s.ln1(x),need_weights=False); x=x+a; return x+s.ml(s.ln2(x))
class GPT(nn.Module):
    def __init__(s, V, L=8, H=8, D=512):
        super().__init__(); s.blk=BLOCK; s.tok=nn.Embedding(V,D); s.pos=nn.Embedding(BLOCK,D)
        s.blocks=nn.ModuleList([Block(D,H) for _ in range(L)]); s.lnf=nn.LayerNorm(D); s.head=nn.Linear(D,V,bias=False)
    def forward(s,x,y=None):
        B,T=x.shape; p=s.pos(torch.arange(T,device=x.device))
        h=s.tok(x)+p
        for b in s.blocks: h=b(h)
        logits=s.head(s.lnf(h))
        return (logits, F.cross_entropy(logits.view(-1,logits.size(-1)),y.view(-1))) if y is not None else (logits,None)
def batch(split):
    d = train_d if split=="train" else val_d
    ix = torch.randint(len(d)-BLOCK-1,(BATCH,))
    x = torch.stack([d[i:i+BLOCK] for i in ix]); y = torch.stack([d[i+1:i+1+BLOCK] for i in ix])
    return x.to(dev), y.to(dev)
model = GPT(V).to(dev)
print("params", sum(p.numel() for p in model.parameters()))
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.1)
ck_path = os.path.join(DRIVE, "bw-colab-latest.pt"); start = 0
if os.path.exists(ck_path):
    ck = torch.load(ck_path, map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start = ck["step"]
    print("resumed at step", start)
log = open(os.path.join(DRIVE, "bw-colab-log.jsonl"), "a")
t0 = time.time()
for step in range(start, STEPS):
    model.train(); x,y = batch("train"); _,loss = model(x,y)
    opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    for g in opt.param_groups: g["lr"] = LR*0.5*(1+math.cos(math.pi*step/STEPS))
    opt.step()
    if step % 50 == 0 or step == STEPS-1:
        model.eval()
        with torch.no_grad():
            vx,vy = batch("val"); _,vl = model(vx,vy)
        row = {"step": step, "train_loss": round(loss.item(),4), "val_loss": round(vl.item(),4),
               "val_perplexity": round(math.exp(vl.item()),2), "elapsed_s": round(time.time()-t0,1), "device": dev}
        log.write(json.dumps(row)+"\n"); log.flush(); print(row)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step+1, "config": {"V":V,"BLOCK":BLOCK}}, ck_path)
print("done")